# Modelado V5 con selección automática de mejor modelo por producto

Este notebook reemplaza el enfoque de entrenar únicamente `RandomForestRegressor`.

Ahora, para cada producto, prueba varios modelos, compara sus métricas en validación cronológica, selecciona el mejor y luego entrena el modelo final con entrenamiento + validación.

El conjunto de prueba se usa solo para medir el desempeño final, no para escoger el modelo.

**NUEVO EN ESTA VERSIÓN:** Se ha agregado un baseline donde se reentrenan y evalúan los modelos "perdedores" usando el conjunto de prueba para compararlos justamente contra el modelo ganador.

## Bloque 1. Montar Drive e importar librerías

Este bloque conecta Google Drive, importa librerías y carga los modelos de scikit-learn que se compararán.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import os
import joblib
import json
from pathlib import Path

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Bloque 2. Rutas y configuración

Ajusta estas rutas según tu entorno.

En tu caso actual, `dataset_preparado_v3.csv` está temporalmente en `/content` y la carpeta `datasets_por_plato` está en `MyDrive`.

In [ ]:
# ============================================================
# 1. RUTAS Y CONFIGURACIÓN
# ============================================================

# Dataset preparado en CSV.
# Si lo mueves a Drive, cambia esta ruta.
INPUT_PREPARADO = "/content/dataset_preparado_v3.csv"

# Carpeta donde están las subcarpetas:
# almuerzo, sopa, fanesca, colada_morada
RUTA_DIVISION = "/content/drive/MyDrive/datasets_por_plato"

# Carpeta donde se guardarán los mejores modelos seleccionados.
# Se usa un nombre genérico porque ya no necesariamente todos serán Random Forest.
OUTPUT_MODELOS = "/content/drive/MyDrive/modelos_mejor_modelo"

# Reportes de salida.
OUTPUT_RESULTADOS = "/content/drive/MyDrive/resultados_modelado_mejor_modelo.xlsx"
OUTPUT_CONFIG_PKL = f"{OUTPUT_MODELOS}/config_entrenamiento.pkl"
OUTPUT_CONFIG_JSON = f"{OUTPUT_MODELOS}/config_entrenamiento.json"

os.makedirs(OUTPUT_MODELOS, exist_ok=True)

PRODUCTOS = ["almuerzo", "sopa", "fanesca", "colada_morada"]

VARIABLES_PREDICTORAS = [
    "anio",
    "mes",
    "dia_mes",
    "dia_semana_num",
    "semana_anio",

    "es_fanesca_temporada",
    "es_colada_temporada",
    "es_inicio_mes",
    "es_quincena",
    "es_fin_mes",

    "es_lunes",
    "es_martes",
    "es_miercoles",
    "es_jueves",
    "es_viernes",

    "mes_sin",
    "mes_cos",
    "dia_semana_sin",
    "dia_semana_cos",

    "tendencia",
    "tendencia_log",
    "crecimiento_anual",

    "preciomenu",
    "preciosopa",
    "fanesca_precio",
    "coladamorada_precio"
]

# Criterio principal de selección.
# Se selecciona el modelo con menor RMSE en validación.
# Si hay empate, se usa menor MAE y luego mayor R².
CRITERIO_PRINCIPAL = "RMSE"

## Bloque 3. Modelos candidatos

Aquí se definen los modelos que se probarán por producto.

Se incluyen modelos simples, modelos lineales regularizados y modelos de árboles.

También se incluye un modelo base (`Baseline_Promedio`) para saber si los modelos realmente mejoran frente a una predicción promedio.

In [ ]:
# ============================================================
# 2. MODELOS CANDIDATOS
# ============================================================

MODELOS_CANDIDATOS = {
    "Baseline_Promedio": DummyRegressor(strategy="mean"),

    "Regresion_Lineal": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=42))
    ]),

    "ElasticNet": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.05, l1_ratio=0.2, random_state=42, max_iter=10000))
    ]),

    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        max_depth=6,
        min_samples_split=8,
        min_samples_leaf=4,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500,
        random_state=42,
        max_depth=8,
        min_samples_split=6,
        min_samples_leaf=3,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=42,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        min_samples_leaf=4
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=42,
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=0.1
    )
}

print("Modelos candidatos:")
for nombre in MODELOS_CANDIDATOS:
    print("-", nombre)

## Bloque 4. Lectura robusta del dataset preparado y configuración base

Este bloque lee el dataset preparado, soportando CSV con separador `;` o `,`.

También valida columnas y rango de fechas.

In [ ]:
# ============================================================
# 3. LECTURA ROBUSTA DEL DATASET PREPARADO
# ============================================================

def leer_dataset_preparado(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    if path.suffix.lower() in [".xlsx", ".xls"]:
        print("Leyendo archivo Excel...")
        return pd.read_excel(path)

    if path.suffix.lower() == ".csv":
        print("Leyendo archivo CSV...")

        intentos = [
            {"sep": ";", "encoding": "utf-8-sig"},
            {"sep": ",", "encoding": "utf-8-sig"},
            {"sep": ";", "encoding": "latin-1"},
            {"sep": ",", "encoding": "latin-1"},
            {"sep": None, "encoding": "utf-8-sig"},
            {"sep": None, "encoding": "latin-1"},
        ]

        ultimo_error = None

        for intento in intentos:
            try:
                if intento["sep"] is None:
                    df_temp = pd.read_csv(
                        path,
                        sep=None,
                        engine="python",
                        encoding=intento["encoding"]
                    )
                else:
                    df_temp = pd.read_csv(
                        path,
                        sep=intento["sep"],
                        engine="python",
                        encoding=intento["encoding"]
                    )

                if df_temp.shape[1] > 1:
                    print(
                        "Archivo leído correctamente con:",
                        f"sep={repr(intento['sep'])},",
                        f"encoding={intento['encoding']}"
                    )
                    return df_temp

            except Exception as e:
                ultimo_error = e

        raise ValueError(
            f"No se pudo leer el CSV correctamente. Último error: {ultimo_error}"
        )

    raise ValueError("Formato no soportado. Usa .csv, .xlsx o .xls.")


df_base = leer_dataset_preparado(INPUT_PREPARADO)

df_base.columns = [str(c).strip() for c in df_base.columns]

if "fecha" not in df_base.columns:
    raise ValueError("El dataset preparado no contiene la columna 'fecha'.")

df_base["fecha"] = pd.to_datetime(df_base["fecha"], errors="coerce")
df_base = df_base.dropna(subset=["fecha"]).sort_values("fecha").reset_index(drop=True)

faltantes = [
    c for c in ["fecha"] + VARIABLES_PREDICTORAS + PRODUCTOS
    if c not in df_base.columns
]

if faltantes:
    raise ValueError(f"Faltan columnas en dataset_preparado: {faltantes}")

print("Dataset preparado leído correctamente.")
print("Filas:", len(df_base))
print("Rango:", df_base["fecha"].min().date(), "a", df_base["fecha"].max().date())

display(df_base.head())

## Bloque 5. Funciones de preparación, métricas e interpretación

Estas funciones se usan durante el entrenamiento:

- Limpian números con coma decimal.
- Preparan train, validación y prueba.
- Calculan métricas.
- Extraen importancia de variables cuando el modelo lo permite.
- Generan textos explicativos para el reporte.

In [ ]:
# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def convertir_numero_seguro(serie):
    # Convierte '0,5' a 0.5 y maneja nulos.
    return (
        serie
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace(["", "nan", "None", "NaN", "NULL"], np.nan)
        .pipe(pd.to_numeric, errors="coerce")
    )


def preparar_conjunto_modelo(df, nombre_conjunto, producto):
    df = df.copy()

    columnas_necesarias = ["fecha"] + VARIABLES_PREDICTORAS + ["demanda"]

    faltantes = [c for c in columnas_necesarias if c not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas en {nombre_conjunto}_{producto}: {faltantes}")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    for col in VARIABLES_PREDICTORAS:
        df[col] = convertir_numero_seguro(df[col])

    df["demanda"] = convertir_numero_seguro(df["demanda"])

    columnas_con_nulos = df[VARIABLES_PREDICTORAS + ["demanda"]].columns[
        df[VARIABLES_PREDICTORAS + ["demanda"]].isna().any()
    ].tolist()

    if columnas_con_nulos:
        print(f"Advertencia en {nombre_conjunto}_{producto}: columnas con nulos convertidos a 0:")
        print(columnas_con_nulos)

    df[VARIABLES_PREDICTORAS] = df[VARIABLES_PREDICTORAS].fillna(0)
    df["demanda"] = df["demanda"].fillna(0)

    return df


def calcular_metricas(y_real, y_pred):
    y_real = np.asarray(y_real, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = mean_absolute_error(y_real, y_pred)
    rmse = mean_squared_error(y_real, y_pred) ** 0.5

    try:
        r2 = r2_score(y_real, y_pred)
    except Exception:
        r2 = np.nan

    if np.sum(y_real) != 0:
        wape = np.sum(np.abs(y_real - y_pred)) / np.sum(y_real) * 100
    else:
        wape = np.nan

    mask_mape = y_real != 0
    if np.any(mask_mape):
        mape = np.mean(np.abs((y_real[mask_mape] - y_pred[mask_mape]) / y_real[mask_mape])) * 100
    else:
        mape = np.nan

    sesgo = np.mean(y_pred - y_real)

    return {
        "MAE": round(float(mae), 3),
        "RMSE": round(float(rmse), 3),
        "R2": round(float(r2), 4) if pd.notna(r2) else np.nan,
        "WAPE_%": round(float(wape), 3) if pd.notna(wape) else np.nan,
        "MAPE_%": round(float(mape), 3) if pd.notna(mape) else np.nan,
        "Sesgo_promedio": round(float(sesgo), 3)
    }


def obtener_estimador_final(modelo):
    if isinstance(modelo, Pipeline):
        return modelo.steps[-1][1]
    return modelo


def extraer_importancias(modelo, producto):
    estimador = obtener_estimador_final(modelo)

    if hasattr(estimador, "feature_importances_"):
        return pd.DataFrame({
            "producto": producto,
            "tipo_importancia": "feature_importances",
            "variable": VARIABLES_PREDICTORAS,
            "importancia": estimador.feature_importances_
        }).sort_values("importancia", ascending=False)

    if hasattr(estimador, "coef_"):
        coef = np.ravel(estimador.coef_)
        if len(coef) == len(VARIABLES_PREDICTORAS):
            return pd.DataFrame({
                "producto": producto,
                "tipo_importancia": "coeficiente_absoluto",
                "variable": VARIABLES_PREDICTORAS,
                "importancia": np.abs(coef),
                "coeficiente": coef
            }).sort_values("importancia", ascending=False)

    return pd.DataFrame()


def explicar_seleccion(producto, mejor_fila):
    return (
        f"Para el producto {producto}, el modelo seleccionado fue {mejor_fila['modelo_candidato']}. "
        f"La selección se realizó usando validación cronológica y el criterio principal fue menor RMSE. "
        f"Este criterio es adecuado porque el RMSE penaliza con mayor fuerza los errores grandes, "
        f"lo cual es importante en predicción de demanda porque un pico mal estimado puede afectar planificación, compras e inventario. "
        f"El modelo seleccionado obtuvo en validación: "
        f"MAE={mejor_fila['MAE']}, RMSE={mejor_fila['RMSE']}, R2={mejor_fila['R2']}, "
        f"WAPE={mejor_fila['WAPE_%']}% y MAPE={mejor_fila['MAPE_%']}%. "
        f"MAE se incluye para interpretar el error promedio en unidades, RMSE para priorizar errores grandes "
        f"y R2 para revisar capacidad explicativa del modelo."
    )

## Bloque 6. Entrenar modelos candidatos y seleccionar el mejor por producto

Para cada producto:

1. Lee train, validación y prueba.
2. Entrena todos los modelos candidatos usando train.
3. Evalúa en validación.
4. Selecciona el mejor con menor RMSE.
5. Reentrena el modelo seleccionado con train + validación.
6. Evalúa el modelo final en prueba.
7. Guarda el `.pkl` final del producto.
8. **NUEVO:** Reentrena y evalúa los modelos perdedores en el conjunto de prueba para crear un baseline comparativo.

In [ ]:
# ============================================================
# 5. ENTRENAMIENTO, COMPARACIÓN Y SELECCIÓN POR PRODUCTO
# ============================================================

comparacion_validacion = []
metricas_finales = []
predicciones_finales = []
importancias_todas = []
seleccion_modelos = []
reporte_texto = []
metricas_estacionales = []

# Comentario: Lista nueva para almacenar el reentrenamiento y prueba del resto de modelos.
comparacion_prueba_todos = [] 

modelo_por_producto = {}
razon_por_producto = {}

for producto in PRODUCTOS:

    print(f"\n{'=' * 80}")
    print(f"Procesando producto: {producto}")
    print(f"{'=' * 80}")

    path_train = f"{RUTA_DIVISION}/{producto}/train_{producto}.xlsx"
    path_val = f"{RUTA_DIVISION}/{producto}/validacion_{producto}.xlsx"
    path_test = f"{RUTA_DIVISION}/{producto}/prueba_{producto}.xlsx"

    if not os.path.exists(path_train):
        raise FileNotFoundError(f"No existe archivo train: {path_train}")
    if not os.path.exists(path_val):
        raise FileNotFoundError(f"No existe archivo validación: {path_val}")
    if not os.path.exists(path_test):
        raise FileNotFoundError(f"No existe archivo prueba: {path_test}")

    train = pd.read_excel(path_train)
    val = pd.read_excel(path_val)
    test = pd.read_excel(path_test)

    train = preparar_conjunto_modelo(train, "train", producto)
    val = preparar_conjunto_modelo(val, "validacion", producto)
    test = preparar_conjunto_modelo(test, "prueba", producto)

    X_train = train[VARIABLES_PREDICTORAS]
    y_train = train["demanda"]

    X_val = val[VARIABLES_PREDICTORAS]
    y_val = val["demanda"]

    X_test = test[VARIABLES_PREDICTORAS]
    y_test = test["demanda"]

    # ------------------------------------------------------------
    # 1. Comparación de modelos candidatos en validación
    # ------------------------------------------------------------

    resultados_producto = []

    for nombre_modelo, modelo_base in MODELOS_CANDIDATOS.items():

        print(f"Entrenando candidato: {nombre_modelo}")

        modelo_candidato = clone(modelo_base)
        modelo_candidato.fit(X_train, y_train)

        pred_val = modelo_candidato.predict(X_val)
        pred_val = np.maximum(pred_val, 0)

        met_val = calcular_metricas(y_val, pred_val)

        fila = {
            "producto": producto,
            "modelo_candidato": nombre_modelo,
            "conjunto": "Validación",
            "registros": len(val),
            "total_real": round(float(np.sum(y_val)), 3),
            "total_predicho": round(float(np.sum(pred_val)), 3),
            **met_val
        }

        resultados_producto.append(fila)
        comparacion_validacion.append(fila)

    df_resultados_producto = pd.DataFrame(resultados_producto)

    # Selección:
    # menor RMSE, luego menor MAE, luego mayor R2.
    df_resultados_producto = df_resultados_producto.sort_values(
        by=["RMSE", "MAE", "R2"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    mejor_fila = df_resultados_producto.iloc[0]
    mejor_nombre = mejor_fila["modelo_candidato"]

    print(f"Mejor modelo para {producto}: {mejor_nombre}")

    # ------------------------------------------------------------
    # 2. Reentrenamiento final con train + validación
    # ------------------------------------------------------------

    train_val = pd.concat([train, val], ignore_index=True)
    X_train_val = train_val[VARIABLES_PREDICTORAS]
    y_train_val = train_val["demanda"]

    modelo_final = clone(MODELOS_CANDIDATOS[mejor_nombre])
    modelo_final.fit(X_train_val, y_train_val)

    # ------------------------------------------------------------
    # 3. Evaluación final en prueba
    # ------------------------------------------------------------

    pred_test = modelo_final.predict(X_test)
    pred_test = np.maximum(pred_test, 0)

    met_test = calcular_metricas(y_test, pred_test)

    fila_final = {
        "producto": producto,
        "modelo_final": mejor_nombre,
        "conjunto": "Prueba",
        "registros": len(test),
        "total_real": round(float(np.sum(y_test)), 3),
        "total_predicho": round(float(np.sum(pred_test)), 3),
        **met_test
    }

    metricas_finales.append(fila_final)

    # ------------------------------------------------------------
    # 3.5 Evaluación de TODOS los modelos en prueba (Baseline)
    # ------------------------------------------------------------
    # Comentario: Este bloque reentrena todos los modelos candidatos
    # con train+val para medir su desempeño equitativamente en prueba.
    print(f"Calculando métricas de prueba para modelos perdedores de {producto}...")
    for nombre_modelo, modelo_base in MODELOS_CANDIDATOS.items():
        # Si es el ganador, copiamos las métricas que ya calculamos en el paso anterior
        if nombre_modelo == mejor_nombre:
            comparacion_prueba_todos.append({
                "producto": producto,
                "modelo_candidato": nombre_modelo,
                "estado": "Ganador",
                **met_test
            })
            continue
        
        # Si es un perdedor, lo clonamos y lo reentrenamos con la data completa
        modelo_perdedor = clone(modelo_base)
        modelo_perdedor.fit(X_train_val, y_train_val)
        
        # Hacemos la predicción en el conjunto de prueba (que no han visto)
        pred_test_perdedor = modelo_perdedor.predict(X_test)
        pred_test_perdedor = np.maximum(pred_test_perdedor, 0)
        
        met_test_perdedor = calcular_metricas(y_test, pred_test_perdedor)
        
        # Guardamos los resultados etiquetándolos como perdedores
        comparacion_prueba_todos.append({
            "producto": producto,
            "modelo_candidato": nombre_modelo,
            "estado": "Perdedor",
            **met_test_perdedor
        })

    # ------------------------------------------------------------
    # 4. Guardar predicciones finales
    # ------------------------------------------------------------

    pred_temp = test[["fecha"]].copy()
    pred_temp["producto"] = producto
    pred_temp["modelo_final"] = mejor_nombre
    pred_temp["conjunto"] = "Prueba"
    pred_temp["demanda_real"] = y_test.values
    pred_temp["demanda_predicha"] = pred_test.round().astype(int)
    pred_temp["error"] = pred_temp["demanda_predicha"] - pred_temp["demanda_real"]
    pred_temp["error_abs"] = pred_temp["error"].abs()

    predicciones_finales.append(pred_temp)

    # ------------------------------------------------------------
    # 5. Guardar importancia de variables
    # ------------------------------------------------------------

    imp = extraer_importancias(modelo_final, producto)
    if not imp.empty:
        imp["modelo_final"] = mejor_nombre
        importancias_todas.append(imp)

    # ------------------------------------------------------------
    # 6. Guardar modelo final
    # ------------------------------------------------------------

    ruta_modelo = f"{OUTPUT_MODELOS}/modelo_{producto}.pkl"
    joblib.dump(modelo_final, ruta_modelo)

    print("Modelo final guardado:", ruta_modelo)

    # ------------------------------------------------------------
    # 7. Reporte de selección
    # ------------------------------------------------------------

    explicacion = explicar_seleccion(producto, mejor_fila)

    seleccion_modelos.append({
        "producto": producto,
        "modelo_seleccionado": mejor_nombre,
        "criterio_principal": "Menor RMSE en validación",
        "criterios_secundarios": "Menor MAE y mayor R2 en caso de empate",
        "MAE_validacion": mejor_fila["MAE"],
        "RMSE_validacion": mejor_fila["RMSE"],
        "R2_validacion": mejor_fila["R2"],
        "WAPE_validacion_%": mejor_fila["WAPE_%"],
        "MAPE_validacion_%": mejor_fila["MAPE_%"],
        "MAE_prueba": met_test["MAE"],
        "RMSE_prueba": met_test["RMSE"],
        "R2_prueba": met_test["R2"],
        "WAPE_prueba_%": met_test["WAPE_%"],
        "MAPE_prueba_%": met_test["MAPE_%"],
        "ruta_modelo": ruta_modelo,
        "explicacion": explicacion
    })

    reporte_texto.append({
        "seccion": f"Selección de modelo - {producto}",
        "contenido": explicacion
    })

    modelo_por_producto[producto] = mejor_nombre
    razon_por_producto[producto] = explicacion

    # ------------------------------------------------------------
    # 8. Evaluación estacional especial si existe
    # ------------------------------------------------------------

    path_train_est = f"{RUTA_DIVISION}/{producto}/train_estacional_{producto}.xlsx"
    path_test_est = f"{RUTA_DIVISION}/{producto}/prueba_estacional_{producto}.xlsx"

    if os.path.exists(path_train_est) and os.path.exists(path_test_est):

        print(f"Evaluación estacional detectada para: {producto}")

        train_est = pd.read_excel(path_train_est)
        test_est = pd.read_excel(path_test_est)

        train_est = preparar_conjunto_modelo(train_est, "train_estacional", producto)
        test_est = preparar_conjunto_modelo(test_est, "prueba_estacional", producto)

        if len(train_est) > 0 and len(test_est) > 0:
            modelo_est = clone(MODELOS_CANDIDATOS[mejor_nombre])
            modelo_est.fit(train_est[VARIABLES_PREDICTORAS], train_est["demanda"])

            y_est = test_est["demanda"]
            pred_est = np.maximum(
                modelo_est.predict(test_est[VARIABLES_PREDICTORAS]),
                0
            )

            met_est = calcular_metricas(y_est, pred_est)

            metricas_estacionales.append({
                "producto": producto,
                "modelo_usado": mejor_nombre,
                "conjunto": "Prueba estacional",
                "registros": len(test_est),
                "total_real": round(float(np.sum(y_est)), 3),
                "total_predicho": round(float(np.sum(pred_est)), 3),
                **met_est
            })

print("\nEntrenamiento y selección automática finalizados.")

## Bloque 7. Crear configuración final

La configuración ahora guarda qué modelo fue seleccionado para cada producto.

Esto es importante porque ya no se asume que todos los productos usan Random Forest.

In [ ]:
# ============================================================
# 6. CONFIGURACIÓN FINAL
# ============================================================

config_entrenamiento = {
    "version": "Version_5_seleccion_automatica",
    "modelo": "Seleccion_automatica_por_producto",
    "enfoque": "un_modelo_por_producto_con_seleccion_automatica",
    "productos": PRODUCTOS,
    "variables_predictoras": VARIABLES_PREDICTORAS,
    "fecha_min_modelo": str(df_base["fecha"].min().date()),
    "fecha_max_modelo": str(df_base["fecha"].max().date()),
    "temporada_colada_morada": {
        "inicio_mes": 10,
        "inicio_dia": 1,
        "fin_mes": 11,
        "fin_dia": 4
    },
    "temporada_fanesca": {
        "meses": [2, 3]
    },
    "base_ciclo_dia_semana": 5,
    "criterio_seleccion": {
        "principal": "Menor RMSE en validación",
        "secundarios": ["Menor MAE", "Mayor R2"],
        "justificacion": (
            "La selección se hace con validación cronológica para evitar fuga de información. "
            "RMSE se usa como criterio principal porque penaliza más los errores grandes, "
            "lo cual es relevante en demanda operativa. MAE ayuda a interpretar el error promedio "
            "en unidades y R2 muestra capacidad explicativa."
        )
    },
    "modelos_candidatos": list(MODELOS_CANDIDATOS.keys()),
    "modelo_por_producto": modelo_por_producto,
    "razon_por_producto": razon_por_producto,
}

joblib.dump(config_entrenamiento, OUTPUT_CONFIG_PKL)

with open(OUTPUT_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(config_entrenamiento, f, ensure_ascii=False, indent=4)

print("Configuración final guardada:")
print(OUTPUT_CONFIG_PKL)
print(OUTPUT_CONFIG_JSON)

display(pd.DataFrame([
    {"producto": p, "modelo_seleccionado": m}
    for p, m in modelo_por_producto.items()
]))

## Bloque 8. Consolidar resultados y generar reporte

Este bloque crea tablas finales:

- Comparación de todos los modelos candidatos.
- Modelo seleccionado por producto.
- Métricas finales en prueba.
- **(NUEVO)** Baseline del desempeño de todos los modelos en el conjunto de prueba.
- Predicciones finales.
- Importancia de variables.
- Evaluación estacional, si aplica.
- Reporte textual con explicación del porqué se eligió cada modelo.

In [ ]:
# ============================================================
# 7. CONSOLIDAR Y EXPORTAR RESULTADOS
# ============================================================

df_comparacion_validacion = pd.DataFrame(comparacion_validacion)
df_metricas_finales = pd.DataFrame(metricas_finales)
df_predicciones_finales = pd.concat(predicciones_finales, ignore_index=True) if predicciones_finales else pd.DataFrame()
df_importancias = pd.concat(importancias_todas, ignore_index=True) if importancias_todas else pd.DataFrame()
df_seleccion_modelos = pd.DataFrame(seleccion_modelos)
df_reporte_texto = pd.DataFrame(reporte_texto)
df_metricas_estacionales = pd.DataFrame(metricas_estacionales)

# Comentario: Convertir la nueva lista de baseline a DataFrame y ordenar
df_comparacion_prueba_todos = pd.DataFrame(comparacion_prueba_todos)
if not df_comparacion_prueba_todos.empty:
    df_comparacion_prueba_todos = df_comparacion_prueba_todos.sort_values(
        ["producto", "estado", "RMSE"], 
        ascending=[True, True, True]
    ).reset_index(drop=True)

# Ordenar comparación por producto y RMSE.
if not df_comparacion_validacion.empty:
    df_comparacion_validacion = df_comparacion_validacion.sort_values(
        ["producto", "RMSE", "MAE", "R2"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)

with pd.ExcelWriter(OUTPUT_RESULTADOS, engine="openpyxl") as writer:
    df_seleccion_modelos.to_excel(writer, sheet_name="seleccion_modelos", index=False)
    df_comparacion_validacion.to_excel(writer, sheet_name="comparacion_validacion", index=False)
    df_metricas_finales.to_excel(writer, sheet_name="metricas_prueba", index=False)

    # Comentario: Exportación de la nueva hoja con el baseline equitativo
    if not df_comparacion_prueba_todos.empty:
        df_comparacion_prueba_todos.to_excel(writer, sheet_name="baseline_prueba_todos", index=False)

    df_predicciones_finales.to_excel(writer, sheet_name="predicciones_prueba", index=False)

    if not df_importancias.empty:
        df_importancias.to_excel(writer, sheet_name="importancia_variables", index=False)

    if not df_metricas_estacionales.empty:
        df_metricas_estacionales.to_excel(writer, sheet_name="metricas_estacionales", index=False)

    df_reporte_texto.to_excel(writer, sheet_name="reporte_texto", index=False)

    metodologia = pd.DataFrame([
        {
            "tema": "Objetivo",
            "descripcion": "Comparar varios modelos por producto y seleccionar automáticamente el mejor según validación cronológica."
        },
        {
            "tema": "Criterio principal",
            "descripcion": "Menor RMSE en validación."
        },
        {
            "tema": "Por qué RMSE",
            "descripcion": "RMSE penaliza más los errores grandes, lo cual es importante cuando un error alto puede afectar planificación de producción, compras o inventario."
        },
        {
            "tema": "Métrica complementaria MAE",
            "descripcion": "MAE muestra el error promedio en unidades de platos, por lo que es fácil de interpretar operativamente."
        },
        {
            "tema": "Métrica complementaria R2",
            "descripcion": "R2 muestra qué tanto el modelo explica la variabilidad de los datos reales. Valores cercanos a 1 son mejores; valores negativos indican bajo desempeño."
        },
        {
            "tema": "Uso de prueba",
            "descripcion": "El conjunto de prueba no se usa para escoger el modelo. Se usa solo para medir el desempeño final del modelo seleccionado."
        },
        {
            "tema": "Productos estacionales",
            "descripcion": "Cuando existen archivos estacionales, se calcula una evaluación adicional para fanesca y colada morada en su temporada."
        }
    ])
    metodologia.to_excel(writer, sheet_name="metodologia", index=False)

print("\nModelado con selección automática finalizado correctamente.")
print("Reporte:", OUTPUT_RESULTADOS)
print("Modelos:", OUTPUT_MODELOS)
print("Configuración:", OUTPUT_CONFIG_PKL)

print("\nModelos seleccionados por producto:")
display(df_seleccion_modelos[[
    "producto",
    "modelo_seleccionado",
    "RMSE_validacion",
    "MAE_validacion",
    "R2_validacion",
    "RMSE_prueba",
    "MAE_prueba",
    "R2_prueba"
]])

print("\nMétricas finales en prueba:")
display(df_metricas_finales)

## Bloque 9. Generar reporte integral de pruebas, resultados y justificación

Este bloque genera un reporte más completo para sustentar el prototipo.

Incluye:

- Todas las pruebas realizadas por producto y por modelo candidato.
- Resultados obtenidos por cada modelo.
- Modelo seleccionado por producto.
- Justificación de selección con base en RMSE, MAE, R², WAPE y MAPE.
- Plan de pruebas del prototipo.
- Trazabilidad entre notebooks, fases del pipeline y evidencias generadas.

Ejecuta este bloque después del Bloque 8.

In [ ]:
# ============================================================
# 8. REPORTE INTEGRAL DE PRUEBAS Y VALIDACIÓN DEL PROTOTIPO
# ============================================================

from pathlib import Path
import json

OUTPUT_REPORTE_INTEGRAL = str(Path(OUTPUT_RESULTADOS).with_name("reporte_integral_pruebas_validacion_prototipo.xlsx"))
OUTPUT_PLAN_PRUEBAS_MD = str(Path(OUTPUT_RESULTADOS).with_name("plan_pruebas_validacion_prototipo.md"))

# ------------------------------------------------------------
# 8.1 Validaciones previas
# ------------------------------------------------------------

objetos_necesarios = [
    "df_comparacion_validacion",
    "df_metricas_finales",
    "df_seleccion_modelos",
    "df_reporte_texto"
]

faltantes_objetos = [obj for obj in objetos_necesarios if obj not in globals()]

if faltantes_objetos:
    raise ValueError(
        "Antes de generar el reporte integral debes ejecutar los bloques anteriores. "
        f"Faltan estos objetos: {faltantes_objetos}"
    )

# ------------------------------------------------------------
# 8.2 Ranking de modelos por producto
# ------------------------------------------------------------

df_ranking_modelos = df_comparacion_validacion.copy()

if not df_ranking_modelos.empty:
    df_ranking_modelos = df_ranking_modelos.sort_values(
        ["producto", "RMSE", "MAE", "R2"],
        ascending=[True, True, True, False]
    ).reset_index(drop=True)

    df_ranking_modelos["ranking_en_producto"] = (
        df_ranking_modelos
        .groupby("producto")
        .cumcount() + 1
    )

# ------------------------------------------------------------
# 8.3 Comparación contra baseline
# ------------------------------------------------------------

df_mejora_vs_baseline = pd.DataFrame()

if not df_comparacion_validacion.empty and "Baseline_Promedio" in df_comparacion_validacion["modelo_candidato"].unique():

    baseline = (
        df_comparacion_validacion[
            df_comparacion_validacion["modelo_candidato"] == "Baseline_Promedio"
        ][["producto", "RMSE", "MAE", "R2"]]
        .rename(columns={
            "RMSE": "RMSE_baseline",
            "MAE": "MAE_baseline",
            "R2": "R2_baseline"
        })
    )

    seleccion_validacion = df_seleccion_modelos[[
        "producto",
        "modelo_seleccionado",
        "RMSE_validacion",
        "MAE_validacion",
        "R2_validacion",
        "WAPE_validacion_%",
        "MAPE_validacion_%"
    ]].copy()

    df_mejora_vs_baseline = seleccion_validacion.merge(
        baseline,
        on="producto",
        how="left"
    )

    df_mejora_vs_baseline["mejora_RMSE_%"] = np.where(
        df_mejora_vs_baseline["RMSE_baseline"] != 0,
        (
            (df_mejora_vs_baseline["RMSE_baseline"] - df_mejora_vs_baseline["RMSE_validacion"])
            / df_mejora_vs_baseline["RMSE_baseline"]
        ) * 100,
        np.nan
    )

    df_mejora_vs_baseline["mejora_MAE_%"] = np.where(
        df_mejora_vs_baseline["MAE_baseline"] != 0,
        (
            (df_mejora_vs_baseline["MAE_baseline"] - df_mejora_vs_baseline["MAE_validacion"])
            / df_mejora_vs_baseline["MAE_baseline"]
        ) * 100,
        np.nan
    )

    df_mejora_vs_baseline["conclusion"] = df_mejora_vs_baseline.apply(
        lambda row: (
            f"El modelo seleccionado ({row['modelo_seleccionado']}) mejora el RMSE frente al baseline en "
            f"{row['mejora_RMSE_%']:.2f}%."
            if pd.notna(row["mejora_RMSE_%"]) and row["mejora_RMSE_%"] > 0
            else f"El modelo seleccionado ({row['modelo_seleccionado']}) no mejora claramente el RMSE frente al baseline; requiere revisión."
        ),
        axis=1
    )

# ------------------------------------------------------------
# 8.4 Justificación por producto
# ------------------------------------------------------------

df_justificacion_modelos = df_seleccion_modelos.copy()

if not df_justificacion_modelos.empty:
    df_justificacion_modelos["justificacion_resumida"] = df_justificacion_modelos.apply(
        lambda row: (
            f"Se eligió {row['modelo_seleccionado']} para {row['producto']} porque obtuvo el menor RMSE "
            f"en validación ({row['RMSE_validacion']}). El MAE fue {row['MAE_validacion']}, "
            f"lo que permite interpretar el error promedio en unidades, y el R² fue {row['R2_validacion']}, "
            f"lo que indica la capacidad explicativa del modelo. La prueba final se mantiene separada para "
            f"medir desempeño sin influir en la selección."
        ),
        axis=1
    )

# ------------------------------------------------------------
# 8.5 Resumen ejecutivo del entrenamiento
# ------------------------------------------------------------

total_pruebas_modelado = len(df_comparacion_validacion)
total_productos = len(PRODUCTOS)
total_modelos_candidatos = len(MODELOS_CANDIDATOS)

df_resumen_ejecutivo = pd.DataFrame([
    {
        "indicador": "Productos evaluados",
        "valor": total_productos,
        "descripcion": "Cada producto tiene un modelo independiente."
    },
    {
        "indicador": "Modelos candidatos por producto",
        "valor": total_modelos_candidatos,
        "descripcion": "Cantidad de algoritmos probados para cada producto."
    },
    {
        "indicador": "Pruebas de validación ejecutadas",
        "valor": total_pruebas_modelado,
        "descripcion": "Total de combinaciones producto-modelo evaluadas en validación."
    },
    {
        "indicador": "Criterio principal",
        "valor": "Menor RMSE en validación",
        "descripcion": "RMSE penaliza más los errores grandes y es útil para demanda operativa."
    },
    {
        "indicador": "Criterios secundarios",
        "valor": "Menor MAE y mayor R²",
        "descripcion": "Usados en caso de empate o resultados cercanos."
    },
    {
        "indicador": "Conjunto de prueba",
        "valor": "No se usa para seleccionar",
        "descripcion": "Se usa únicamente para medir desempeño final del modelo elegido."
    }
])

# ------------------------------------------------------------
# 8.6 Plan de pruebas del prototipo
# ------------------------------------------------------------

plan_pruebas = [
    {
        "ID": "CP-01",
        "Fase": "Limpieza de datos",
        "Caso de prueba": "Carga del dataset original",
        "Descripción": "Verificar que el archivo original pueda cargarse y normalizarse correctamente.",
        "Entrada": "dataset_original.xlsx o dataset_limpio.xlsx",
        "Criterio de aceptación": "El archivo se lee sin errores y se identifican columnas esenciales.",
        "Evidencia": "dataset_limpio.xlsx / logs de limpieza",
        "Tipo": "Funcional",
        "Estado esperado": "Aprobado si el archivo se procesa."
    },
    {
        "ID": "CP-02",
        "Fase": "Limpieza de datos",
        "Caso de prueba": "Validación de columnas obligatorias",
        "Descripción": "Confirmar la existencia de Fecha, Tipo_plato y Clientes.",
        "Entrada": "Archivo con estructura válida e inválida",
        "Criterio de aceptación": "El sistema detecta columnas faltantes y muestra error claro.",
        "Evidencia": "Reporte de validación / mensaje de error",
        "Tipo": "Validación",
        "Estado esperado": "Aprobado si detecta faltantes."
    },
    {
        "ID": "CP-03",
        "Fase": "Limpieza de datos",
        "Caso de prueba": "Tratamiento de fechas inválidas",
        "Descripción": "Evaluar registros con fechas incorrectas o nulas.",
        "Entrada": "Archivo con fechas inválidas",
        "Criterio de aceptación": "El sistema excluye o reporta registros inválidos.",
        "Evidencia": "Reporte de limpieza",
        "Tipo": "Validación",
        "Estado esperado": "Aprobado si no pasan fechas inválidas al modelado."
    },
    {
        "ID": "CP-04",
        "Fase": "Preparación de datos",
        "Caso de prueba": "Generación de variables predictoras",
        "Descripción": "Validar creación de variables temporales, estacionales, cíclicas, tendencia y precios.",
        "Entrada": "dataset_limpio.xlsx",
        "Criterio de aceptación": "Todas las variables predictoras definidas están presentes y sin nulos críticos.",
        "Evidencia": "dataset_preparado.xlsx / auditoria_preparacion_datos.xlsx",
        "Tipo": "Funcional",
        "Estado esperado": "Aprobado si dataset_preparado contiene todas las variables."
    },
    {
        "ID": "CP-05",
        "Fase": "Preparación de datos",
        "Caso de prueba": "Control de productos estacionales",
        "Descripción": "Verificar que fanesca y colada morada se controlen fuera de temporada.",
        "Entrada": "Fechas dentro y fuera de temporada",
        "Criterio de aceptación": "Fuera de temporada la demanda estacional se controla según regla de negocio.",
        "Evidencia": "dataset_preparado.xlsx",
        "Tipo": "Regla de negocio",
        "Estado esperado": "Aprobado si se aplican reglas estacionales."
    },
    {
        "ID": "CP-06",
        "Fase": "División del dataset",
        "Caso de prueba": "Creación de datasets por producto",
        "Descripción": "Generar dataset independiente para almuerzo, sopa, fanesca y colada morada.",
        "Entrada": "dataset_preparado.xlsx",
        "Criterio de aceptación": "Se crean carpetas por producto con train, validación y prueba.",
        "Evidencia": "datasets_por_plato/",
        "Tipo": "Funcional",
        "Estado esperado": "Aprobado si existen todos los archivos."
    },
    {
        "ID": "CP-07",
        "Fase": "División del dataset",
        "Caso de prueba": "Split cronológico",
        "Descripción": "Validar que train, validación y prueba respeten el orden temporal.",
        "Entrada": "dataset_preparado.xlsx",
        "Criterio de aceptación": "Train contiene fechas antiguas, validación fechas intermedias y prueba fechas recientes.",
        "Evidencia": "resumen_division_dataset.xlsx",
        "Tipo": "Validación temporal",
        "Estado esperado": "Aprobado si no hay fuga temporal."
    },
    {
        "ID": "CP-08",
        "Fase": "Modelado",
        "Caso de prueba": "Prueba de modelos candidatos",
        "Descripción": "Entrenar y evaluar varios modelos candidatos por producto.",
        "Entrada": "train y validación por producto",
        "Criterio de aceptación": "Cada modelo candidato genera métricas de validación.",
        "Evidencia": "comparacion_validacion",
        "Tipo": "Modelado",
        "Estado esperado": "Aprobado si se generan métricas por producto-modelo."
    },
    {
        "ID": "CP-09",
        "Fase": "Modelado",
        "Caso de prueba": "Selección automática del mejor modelo",
        "Descripción": "Seleccionar el modelo con menor RMSE en validación, usando MAE y R² como apoyo.",
        "Entrada": "Resultados de validación",
        "Criterio de aceptación": "Se selecciona un modelo por producto y se justifica la elección.",
        "Evidencia": "seleccion_modelos / reporte_texto",
        "Tipo": "Modelado",
        "Estado esperado": "Aprobado si existe modelo seleccionado por producto."
    },
    {
        "ID": "CP-10",
        "Fase": "Modelado",
        "Caso de prueba": "Evaluación final en prueba",
        "Descripción": "Medir desempeño del modelo seleccionado en datos no usados para selección.",
        "Entrada": "Conjunto de prueba",
        "Criterio de aceptación": "Se calculan MAE, RMSE, R², WAPE y MAPE en prueba.",
        "Evidencia": "metricas_prueba",
        "Tipo": "Evaluación",
        "Estado esperado": "Aprobado si existen métricas finales."
    },
    {
        "ID": "CP-11",
        "Fase": "Modelado",
        "Caso de prueba": "Evaluación de productos estacionales",
        "Descripción": "Validar desempeño adicional en temporada para fanesca y colada morada.",
        "Entrada": "train_estacional y prueba_estacional",
        "Criterio de aceptación": "Se generan métricas estacionales cuando existen archivos estacionales.",
        "Evidencia": "metricas_estacionales",
        "Tipo": "Evaluación estacional",
        "Estado esperado": "Aprobado si aplica."
    },
    {
        "ID": "CP-12",
        "Fase": "Persistencia",
        "Caso de prueba": "Generación de modelos PKL y configuración",
        "Descripción": "Guardar modelos entrenados y configuración del entrenamiento.",
        "Entrada": "Modelos seleccionados",
        "Criterio de aceptación": "Existen modelo_*.pkl, config_entrenamiento.pkl y config_entrenamiento.json.",
        "Evidencia": "modelos_mejor_modelo/",
        "Tipo": "Persistencia",
        "Estado esperado": "Aprobado si todos los archivos se generan."
    },
    {
        "ID": "CP-13",
        "Fase": "App Streamlit",
        "Caso de prueba": "Carga de modelos en app",
        "Descripción": "Validar que la app pueda cargar los PKL y la configuración.",
        "Entrada": "Carpeta modelos/",
        "Criterio de aceptación": "La sección Diagnóstico de modelos cargados muestra todos los productos.",
        "Evidencia": "validacion_app_v5_multimodelo.ipynb / captura de app",
        "Tipo": "Integración",
        "Estado esperado": "Aprobado si la app carga modelos."
    },
    {
        "ID": "CP-14",
        "Fase": "App Streamlit",
        "Caso de prueba": "Carga de archivo Excel/CSV",
        "Descripción": "Verificar que el usuario pueda cargar un archivo histórico.",
        "Entrada": "Archivo CSV/XLSX válido",
        "Criterio de aceptación": "La app muestra vista previa y permite ejecutar predicción.",
        "Evidencia": "Vista previa del archivo",
        "Tipo": "Funcional",
        "Estado esperado": "Aprobado si procesa archivo."
    },
    {
        "ID": "CP-15",
        "Fase": "App Streamlit",
        "Caso de prueba": "Backtesting con archivo histórico",
        "Descripción": "Subir un archivo de un periodo pasado y comparar predicción contra valores reales.",
        "Entrada": "Excel/CSV con Clientes",
        "Criterio de aceptación": "La app genera predicciones y métricas MAE, RMSE y R².",
        "Evidencia": "Sección Evaluación de calidad por producto",
        "Tipo": "Validación funcional",
        "Estado esperado": "Aprobado si calcula métricas."
    },
    {
        "ID": "CP-16",
        "Fase": "App Streamlit",
        "Caso de prueba": "Predicción futura 6 meses",
        "Descripción": "Generar calendario futuro y estimar demanda sin archivo cargado.",
        "Entrada": "Fecha inicial y precios futuros",
        "Criterio de aceptación": "La app genera predicciones diarias, semanales y mensuales sin métricas reales.",
        "Evidencia": "Tablas y gráficos de predicción futura",
        "Tipo": "Funcional",
        "Estado esperado": "Aprobado si genera proyección."
    },
    {
        "ID": "CP-17",
        "Fase": "App Streamlit",
        "Caso de prueba": "Consulta por día, semana y mes",
        "Descripción": "Validar filtros de consulta de resultados.",
        "Entrada": "Predicciones generadas",
        "Criterio de aceptación": "La tabla y gráficos cambian según el nivel seleccionado.",
        "Evidencia": "Consulta por periodo",
        "Tipo": "UX",
        "Estado esperado": "Aprobado si filtros responden."
    },
    {
        "ID": "CP-18",
        "Fase": "App Streamlit",
        "Caso de prueba": "Descarga de resultados",
        "Descripción": "Validar exportación de predicciones y métricas.",
        "Entrada": "Predicciones generadas",
        "Criterio de aceptación": "Se descarga Excel con predicciones, agregados, insights, métricas y diagnóstico.",
        "Evidencia": "predicciones_demanda.xlsx",
        "Tipo": "Salida",
        "Estado esperado": "Aprobado si descarga correctamente."
    }
]

df_plan_pruebas = pd.DataFrame(plan_pruebas)

# ------------------------------------------------------------
# 8.7 Trazabilidad de notebooks y evidencias
# ------------------------------------------------------------

df_trazabilidad = pd.DataFrame([
    {
        "Notebook / componente": "Limpieza_de_datos.ipynb",
        "Propósito": "Normalizar datos originales, corregir formatos, depurar fechas, productos, clientes y precios.",
        "Entrada principal": "dataset_original.xlsx",
        "Salida / evidencia": "dataset_limpio.xlsx"
    },
    {
        "Notebook / componente": "preparacion_datos_v5.ipynb",
        "Propósito": "Crear variables predictoras, controlar estacionalidad y generar dataset listo para modelado.",
        "Entrada principal": "dataset_limpio.xlsx",
        "Salida / evidencia": "dataset_preparado.xlsx / auditoria_preparacion_datos.xlsx"
    },
    {
        "Notebook / componente": "division_dataset_v5.ipynb",
        "Propósito": "Dividir cronológicamente datasets independientes por producto.",
        "Entrada principal": "dataset_preparado.xlsx",
        "Salida / evidencia": "datasets_por_plato/ / resumen_division_dataset.xlsx"
    },
    {
        "Notebook / componente": "modelado_v5.ipynb",
        "Propósito": "Probar modelos candidatos, seleccionar el mejor por producto y generar PKL finales.",
        "Entrada principal": "datasets_por_plato/",
        "Salida / evidencia": "modelos_mejor_modelo/ / resultados_modelado_mejor_modelo.xlsx"
    },
    {
        "Notebook / componente": "validacion_app_v5_multimodelo.ipynb",
        "Propósito": "Validar que la app pueda cargar configuración y modelos, y ejecutar una predicción de prueba.",
        "Entrada principal": "modelos_mejor_modelo/",
        "Salida / evidencia": "reporte_validacion_app_multimodelo.xlsx / modelos_para_app.zip"
    },
    {
        "Notebook / componente": "app.py",
        "Propósito": "Permitir al usuario cargar archivos, ejecutar predicción, consultar resultados y descargar Excel.",
        "Entrada principal": "CSV/XLSX usuario o parámetros de predicción futura",
        "Salida / evidencia": "Interfaz Streamlit / predicciones_demanda.xlsx"
    }
])

# ------------------------------------------------------------
# 8.8 Texto metodológico para informe
# ------------------------------------------------------------

texto_intro = (
    "Con el propósito de verificar el correcto funcionamiento del prototipo de predicción de demanda, "
    "se definió un plan de pruebas orientado a validar la integridad de los datos de entrada, "
    "el tratamiento de errores, la generación de variables predictoras, la división cronológica de los datos, "
    "el entrenamiento de modelos, la selección del mejor modelo por producto y la integración con la interfaz Streamlit. "
    "El plan incluye casos con entradas válidas e inválidas, criterios de aceptación y evidencias de ejecución, "
    "con el fin de asegurar la trazabilidad y confiabilidad del sistema desarrollado."
)

texto_metodologia_modelado = (
    "La selección de modelos se realizó mediante validación cronológica. Para cada producto se entrenaron varios "
    "modelos candidatos y se evaluaron en un conjunto de validación independiente. El criterio principal de selección "
    "fue el menor RMSE, debido a que esta métrica penaliza con mayor fuerza los errores grandes, lo cual es relevante "
    "en predicción de demanda porque un pico mal estimado puede afectar producción, compras o inventario. "
    "Como métricas complementarias se utilizaron MAE, para interpretar el error promedio en unidades; R², para revisar "
    "la capacidad explicativa del modelo; WAPE, para evaluar el error ponderado frente al total real; y MAPE, para "
    "obtener una referencia porcentual cuando existen valores reales distintos de cero."
)

texto_prueba_final = (
    "El conjunto de prueba no fue utilizado para escoger el modelo, sino únicamente para evaluar el desempeño final "
    "del modelo seleccionado. Esta separación permite evitar fuga de información y proporciona una medición más realista "
    "del comportamiento esperado del modelo ante datos no utilizados durante el entrenamiento ni durante la selección."
)

df_textos_informe = pd.DataFrame([
    {"apartado": "Texto introductorio sugerido", "contenido": texto_intro},
    {"apartado": "Metodología de selección de modelos", "contenido": texto_metodologia_modelado},
    {"apartado": "Uso del conjunto de prueba", "contenido": texto_prueba_final}
])

# ------------------------------------------------------------
# 8.9 Exportar Excel integral
# ------------------------------------------------------------

with pd.ExcelWriter(OUTPUT_REPORTE_INTEGRAL, engine="openpyxl") as writer:
    df_resumen_ejecutivo.to_excel(writer, sheet_name="resumen_ejecutivo", index=False)
    df_plan_pruebas.to_excel(writer, sheet_name="plan_pruebas", index=False)
    df_trazabilidad.to_excel(writer, sheet_name="trazabilidad", index=False)
    df_textos_informe.to_excel(writer, sheet_name="textos_informe", index=False)

    df_ranking_modelos.to_excel(writer, sheet_name="ranking_modelos", index=False)
    df_comparacion_validacion.to_excel(writer, sheet_name="todas_las_pruebas", index=False)
    df_seleccion_modelos.to_excel(writer, sheet_name="modelos_seleccionados", index=False)
    df_justificacion_modelos.to_excel(writer, sheet_name="justificacion_modelos", index=False)
    df_metricas_finales.to_excel(writer, sheet_name="metricas_prueba", index=False)

    if not df_mejora_vs_baseline.empty:
        df_mejora_vs_baseline.to_excel(writer, sheet_name="mejora_vs_baseline", index=False)

    if not df_metricas_estacionales.empty:
        df_metricas_estacionales.to_excel(writer, sheet_name="metricas_estacionales", index=False)

    if not df_importancias.empty:
        df_importancias.to_excel(writer, sheet_name="importancia_variables", index=False)

# ------------------------------------------------------------
# 8.10 Exportar versión Markdown para pegar en documento
# ------------------------------------------------------------

tabla_plan_md = df_plan_pruebas[[
    "ID",
    "Fase",
    "Caso de prueba",
    "Criterio de aceptación",
    "Evidencia"
]].to_markdown(index=False)

tabla_modelos_md = df_seleccion_modelos[[
    "producto",
    "modelo_seleccionado",
    "RMSE_validacion",
    "MAE_validacion",
    "R2_validacion",
    "RMSE_prueba",
    "MAE_prueba",
    "R2_prueba"
]].to_markdown(index=False)

contenido_md = (
    "# Plan de pruebas y validación del prototipo\n\n"
    "## Texto introductorio sugerido\n\n"
    + texto_intro
    + "\n\n## Plan de pruebas\n\n"
    + tabla_plan_md
    + "\n\n## Selección de modelos\n\n"
    + texto_metodologia_modelado
    + "\n\n## Modelos seleccionados y métricas principales\n\n"
    + tabla_modelos_md
    + "\n\n## Validación final\n\n"
    + texto_prueba_final
)

Path(OUTPUT_PLAN_PRUEBAS_MD).write_text(contenido_md, encoding="utf-8")

print("Reporte integral generado correctamente:")
print(OUTPUT_REPORTE_INTEGRAL)

print("Archivo Markdown generado correctamente:")
print(OUTPUT_PLAN_PRUEBAS_MD)

print("\nResumen de modelos seleccionados:")
display(df_seleccion_modelos[[
    "producto",
    "modelo_seleccionado",
    "RMSE_validacion",
    "MAE_validacion",
    "R2_validacion",
    "RMSE_prueba",
    "MAE_prueba",
    "R2_prueba"
]])

print("\nPlan de pruebas:")
display(df_plan_pruebas)

## Bloque 10. Texto listo para pegar en el documento

Este bloque imprime un texto resumido que puedes usar en tu documento de tesis o informe.

In [ ]:
# ============================================================
# 9. TEXTO LISTO PARA PEGAR EN EL DOCUMENTO
# ============================================================

print("7.1.3 Plan de pruebas y validación del prototipo")
print()
print(texto_intro)
print()
print("Tabla X")
print("Plan de pruebas y validación del prototipo de predicción de demanda")
print()
display(df_plan_pruebas[[
    "ID",
    "Fase",
    "Caso de prueba",
    "Descripción",
    "Criterio de aceptación",
    "Evidencia"
]])

print()
print("Texto sugerido para explicar la selección de modelos:")
print()
print(texto_metodologia_modelado)
print()
print("Tabla X")
print("Resumen de modelos seleccionados por producto")
display(df_seleccion_modelos[[
    "producto",
    "modelo_seleccionado",
    "RMSE_validacion",
    "MAE_validacion",
    "R2_validacion",
    "RMSE_prueba",
    "MAE_prueba",
    "R2_prueba"
]])